In [ ]:
import ee
import geemap
# import geopandas as gpd
import sys
from datetime import datetime
import calendar

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
# Definir área de estudio - polígono de coordenadas proporcionadas
table_coords = [
    [-53.55596852754523, -33.62760434283625],
    [-53.51202321504523, -33.62989126644306],
    [-53.42825246309211, -33.53149875353305],
    [-53.38156056856086, -33.458203657841764],
    [-53.43237233613898, -33.26665740930274],
    [-53.33074880098273, -33.12300888910924],
    [-53.24697804902961, -33.12760928990196],
    [-53.13848805879523, -32.99409987632992],
    [-53.12475514863898, -32.85635062675909],
    [-53.012831930865545, -32.82808267148409],
    [-52.86245656465461, -32.94340543371093],
    [-52.73886037324836, -32.911130190517156],
    [-52.60427785371711, -32.669269097165845],
    [-52.55071950410773, -32.50264484289281],
    [-52.66195607637336, -32.3043768720441],
    [-52.58367848848273, -32.128936636373695],
    [-52.74435353731086, -32.1277736527462],
    [-52.86245656465461, -32.33687160651932],
    [-52.86108327363898, -32.40994210748778],
    [-53.01077199434211, -32.47947810330707],
    [-53.07943654512336, -32.59640909893903],
    [-53.21813893770148, -32.64614536707469],
    [-53.23599172090461, -32.70047664235717],
    [-53.34173512910773, -32.754774861808656],
    [-53.37194753145148, -32.87595983489818],
    [-53.53536916231086, -32.93360883667928],
    [-53.69329762910773, -33.1132322371933],
    [-53.69467092012336, -33.23277760490648],
    [-53.63012624238898, -33.2982284459622],
    [-53.61021352266242, -33.391726409857014],
    [-53.619826559771795, -33.44617275773148],
    [-53.60128713106086, -33.53951163942829],
    [-53.55596852754523, -33.62760434283625]
]

In [ ]:
table = ee.Geometry.Polygon(table_coords)

In [ ]:
# Recibir fecha como YYYY-MM
fecha_str = '2026-02'

In [ ]:
# Calcular primer y último día del mes
anio, mes = map(int, fecha_str.split('-'))
primer_dia = datetime(anio, mes, 1).strftime('%Y-%m-%d')
ultimo_dia = datetime(anio, mes, calendar.monthrange(anio, mes)[1]).strftime('%Y-%m-%d')

In [ ]:
# Seteos básicos
start_date = primer_dia
end_date = ultimo_dia
year = ee.Date(start_date).get('year')

In [ ]:
# Colección de imágenes Sentinel-2
inicial_filter = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                  .filterDate(start_date, end_date)
                  .filterBounds(table)
                  .map(lambda img: img.clip(table)))

# Colección de probabilidad de nubes
sentinel_clouds = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
                   .filterDate(start_date, end_date)
                   .filterBounds(table))

In [ ]:
#Calcula el porcentaje de nubes en la región de interés.
def cloud_percent(image):
    # Recortar imagen a la región de interés
    image_interest = image.clip(table)

    # Crear máscara de nubes basada en el umbral
    is_cloud = image_interest.gt(cloud_probability_threshold).rename('cloud_mask')

    # Contar píxeles con nubes
    cloud_pixels = is_cloud.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=table,
        scale=10,
        maxPixels=1e9
    ).get('cloud_mask')

    # Contar total de píxeles válidos
    total_pixels = is_cloud.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=table,
        scale=10,
        maxPixels=1e9
    ).get('cloud_mask')

    # Calcular porcentaje de nubes
    cloud_percentage = ee.Number(cloud_pixels).divide(total_pixels).multiply(100)

    return image.set('cloudPercentage', cloud_percentage)
# Asocia el porcentaje de nubes a las imágenes ópticas Sentinel-2.
def add_cloud_percentage_property(image):
    # Obtener el índice de la imagen
    index = image.get('system:index')

    # Buscar la imagen correspondiente de probabilidad de nubes
    cloud_image = (percent_cloud_collection
                   .filterMetadata('system:index', 'equals', index)
                   .first())

    # Obtener el porcentaje de nubes calculado
    cloud_percentage = ee.Number(cloud_image.get('cloudPercentage'))

    return image.set('cloudPercentage', cloud_percentage)

#Verifica la cobertura de píxeles válidos en cada composite mensual.
def check_coverage(image):
    # Contar píxeles válidos usando banda B4 como referencia
    valid_pixels = (image.select('B4')
                    .unmask(-9999)
                    .neq(-9999)
                    .reduceRegion(
                        reducer=ee.Reducer.sum(),
                        geometry=table,
                        scale=10,
                        maxPixels=1e9
                    ).get('B4'))

    # Contar total de píxeles en la región
    total_pixels = (ee.Image(1)
                    .reduceRegion(
                        reducer=ee.Reducer.sum(),
                        geometry=table,
                        scale=10,
                        maxPixels=1e9
                    ).get('constant'))

    # Calcular porcentaje de cobertura
    coverage = ee.Number(valid_pixels).divide(total_pixels).multiply(100)

    return image.set('coverage_percent', coverage)

def add_area_properties(feature):
    """
    Función auxiliar para calcular áreas de polígonos de agua.
    Debe estar definida fuera de calcular_espejos para evitar problemas de serialización.
    """
    area_m2 = feature.geometry().area(1)
    area_ha = area_m2.divide(10000)
    return feature.set({
        'area_m2': area_m2,
        'area_ha': area_ha
    })

#Calcula cuerpos de agua usando múltiples índices espectrales y vectoriza los resultados.
def calcular_espejos(image):
    # Calcular índice AWEI (Automated Water Extraction Index)
    awei = image.expression(
        '4*(GREEN - SWIR1) - (0.25*NIR + 2.75*SWIR2)', {
            'GREEN': image.select('B3').multiply(0.0001),
            'NIR': image.select('B8').multiply(0.0001),
            'SWIR1': image.select('B11').multiply(0.0001),
            'SWIR2': image.select('B12').multiply(0.0001)
        })
    # Calcular índices normalizados
    ndwi = image.normalizedDifference(['B3', 'B8'])    # Normalized Difference Water Index
    mndwi = image.normalizedDifference(['B3', 'B11'])  # Modified NDWI

    # Crear máscaras binarias (agua = valores negativos)
    NDWI = ndwi.lt(0)
    MNDWI = mndwi.lt(0) 
    AWEI = awei.lt(0)

    # Combinar las tres máscaras (agua debe cumplir los 3 criterios)
    water_combine = AWEI.And(NDWI).And(MNDWI).rename('w')
    water = water_combine.updateMask(water_combine.eq(0))

    # Vectorizar los cuerpos de agua
    water_vectors = water.reduceToVectors(
        geometry=table,
        crs='EPSG:32721',
        maxPixels=1e10,
        scale=20,
        geometryType='polygon',
        eightConnected=True
    )

    # Agregar propiedades de área a cada polígono
    water_vectors_con_area = water_vectors.map(add_area_properties)

    # Filtrar cuerpos de agua grandes
    grandes = water_vectors_con_area.filter(ee.Filter.gt('area_ha', 100000))

    # Calcular área total, retornar -999 si no hay cuerpos grandes
    area_total_ha = ee.Algorithms.If(
        grandes.size().gt(0),
        grandes.aggregate_sum('area_ha'),
        -999
    )

    return image.set('areaTotal_ha', area_total_ha)



In [ ]:
# Procesamiento de colecciones - Aplicación de filtros de nubosidad
cloud_probability_threshold = 70
percent_cloud_collection = sentinel_clouds.map(cloud_percent)

sentinel_collection_with_cloud_percentage = inicial_filter.map(add_cloud_percentage_property)

cloud_threshold = 5
sentinel_filter_collection = (sentinel_collection_with_cloud_percentage
                              .filter(ee.Filter.lt('cloudPercentage', cloud_threshold)))

print("\n=== RESULTADOS DEL PROCESAMIENTO ===")
print(f"Imágenes originales encontradas: {inicial_filter.size().getInfo()}")
print(f"Imágenes de probabilidad de nubes: {sentinel_clouds.size().getInfo()}")
print(f"Imágenes con nubosidad < {cloud_threshold}%: {sentinel_filter_collection.size().getInfo()}")